In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# Feature engineering

df = pd.read_csv('/content/train.csv')

df = df.drop(['Id'], axis=1)

df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 82 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   Alley          91 non-null     object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Neighborhood   1460 non-null   object 
 12  Condition1     1460 non-null   object 
 13  Condition2     1460 non-null   object 
 14  BldgType       1460 non-null   object 
 15  HouseStyle     1460 non-null   object 
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuil

In [3]:
#splitting

X, y = df.drop('SalePrice', axis=1), df['SalePrice']
y = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
#make a custom age transformer

from sklearn.base import BaseEstimator, TransformerMixin

class AgeFeatureCreator(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()

        if 'YearBuilt' in X_copy.columns and 'YrSold' in X_copy.columns:
            X_copy['HouseAge'] = X_copy['YrSold'] - X_copy['YearBuilt']
            X_copy.drop(['YearBuilt', 'YrSold'], axis=1, inplace=True)

        if 'GarageYrBlt' in X_copy.columns:
            X_copy['GarageAge'] = X['YrSold'] - X_copy['GarageYrBlt'] # Use the original YrSold before dropping
            X_copy.drop('GarageYrBlt', axis=1, inplace=True)

        return X_copy

In [5]:
# Define the quality mapping
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}

ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'FireplaceQu',
                'GarageQual', 'GarageCond', 'PoolQC']

for col in ordinal_cols:
    X_train[col] = X_train[col].map(quality_map).fillna(0)
    X_test[col] = X_test[col].map(quality_map).fillna(0)

In [6]:
#Pipelining for catboost

numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

cat_numerical_transformer = Pipeline(steps=[
    ('age_creator', AgeFeatureCreator()),
    ('imputer', SimpleImputer(strategy='median'))
])

cat_categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', cat_numerical_transformer, numerical_cols),
        ('cat', cat_categorical_transformer, categorical_cols)
    ]
)

In [7]:
#Pipelining for Lasso

lasso_numerical_transformer = Pipeline(steps=[
    ('age_creator', AgeFeatureCreator()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

lasso_categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

lasso_preprocessor = ColumnTransformer(
    transformers=[
        ('num', lasso_numerical_transformer, numerical_cols),
        ('cat', lasso_categorical_transformer, categorical_cols)
    ]
)

In [8]:
!pip install catboost

In [9]:
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline(steps=[
    ('preprocessor', lasso_preprocessor),
    ('regressor', Lasso(alpha=0.0005, random_state=42))
])

num_features_after_age_creator = len(numerical_cols) - 1

cat_features_indices = list(range(num_features_after_age_creator, num_features_after_age_creator + len(categorical_cols)))

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', CatBoostRegressor(
          depth=4,
          learning_rate=0.04,
          iterations=3000,
          l2_leaf_reg=3,
          loss_function='RMSE',
          random_state=42,
          verbose=0,
          cat_features=cat_features_indices
    ))
    ]
)

pipeline.fit(X_train, y_train)
lasso_pipeline.fit(X_train, y_train)

catboost_pred = pipeline.predict(X_test)
lasso_pred = lasso_pipeline.predict(X_test)

final_pred = (0.7 * catboost_pred) + (0.3 * lasso_pred)

rmse_ensemble = np.sqrt(mean_squared_error(y_test, final_pred))
r2_ensemble = r2_score(y_test, final_pred)

print(f"Ensemble RMSE: {rmse_ensemble:.5f}")
print(f"Ensemble R^2: {r2_ensemble:.5f}")

Ensemble RMSE: 0.12785
Ensemble R^2: 0.91241


In [10]:
import joblib

filename = 'ensemble_model_v1.joblib'
full_model_package = {
    'catboost_pipe': pipeline,
    'lasso_pipe': lasso_pipeline,
    'catboost_weight': 0.7,
    'lasso_weight': 0.3
}
joblib.dump(full_model_package, filename)
print(f"Pipeline successfully saved to {filename}")

Pipeline successfully saved to house_price_pipeline_v1.joblib
